# Create zarr storing unstitched tiles using ngff-zarr

Same as `create-tiles.ipynb`, but written with [ngff-zarr](https://github.com/fideus-labs/ngff-zarr) instead of ome-zarr-py.

Related image.sc thread: https://forum.image.sc/t/separate-tiles-to-ome-zarr/109071/60

In [ ]:
from pathlib import Path
import tifffile
import ngff_zarr as nz
from ngff_zarr.v06.zarr_metadata import CoordinateSystem, CoordinateSystemIdentifier, Translation

## Source data

In [ ]:
source_path = Path('../data/example_cells')

filenames = sorted(source_path.glob('*.tiff'))
filenames

## Function to read tiff

In [ ]:
def read_tiff(filename):
    with tifffile.TiffFile(filename) as tif:
        page = tif.pages.first
        data = page.asarray()[..., :3]      # drop (constant) alpha channel
        # XPosition/YPosition hold the aligned tile offset in pixels as a rational (num, denom)
        position = {dim: num / denom for dim in 'yx'
                    for num, denom in [page.tags[f'{dim.upper()}Position'].value]}
    return data.transpose(2, 0, 1), position    # yxc -> cyx

## Read source images and store as ome-zarr

In [ ]:
dim_order = 'cyx'
pixel_size = {'c': 1, 'y': 0.004, 'x': 0.004}
unit = 'micrometer'
scale_factors = [2, 4]
chunks = (3, 820, 820)
channel_names = ['R', 'G', 'B']
channel_colors = ['FF0000', '00FF00', '0000FF']

output_path = '../output/tiles-ngff.ome.zarr'


images = {}
transforms = []
for filename in filenames:
    data, position = read_tiff(filename)
    image = nz.to_ngff_image(data, dims=dim_order, scale=pixel_size, name=filename.stem,
                             axes_units={dim: unit for dim in 'yx'}, channel_names=channel_names)
    multiscales = nz.to_multiscales(image, scale_factors=scale_factors, chunks=chunks)
    # to_multiscales doesn't add omero metadata; without it viewers show the channels as grey/default colors
    multiscales.metadata.omero = nz.Omero(channels=[
        nz.OmeroChannel(color=color, window=nz.OmeroWindow(min=0, max=255, start=0, end=255), label=name)
        for name, color in zip(channel_names, channel_colors)])
    translation = [position.get(dim, 0) * pixel_size.get(dim, 1) for dim in dim_order]
    images[filename.stem] = multiscales
    transforms.append(Translation(
        translation=translation,
        input=CoordinateSystemIdentifier(path=filename.stem, name=multiscales.metadata.to_version('0.6').coordinateSystems[0].name),
        output=CoordinateSystemIdentifier(name='stitched'),
    ))
    print(f'{filename} -> {filename.stem} @ {translation}')

# reuse the tiles' physical axes for the scene coordinate system
first_image = next(iter(images.values()))
cs = CoordinateSystem(name='stitched', axes=first_image.metadata.to_version('0.6').coordinateSystems[0].axes)
print('writing scene')
nz.NgffScene(images=images, coordinateTransformations=transforms, coordinateSystems=[cs]).to_ome_zarr(output_path, version='0.6')
print('done')